# EDA — Comunas y barrios de Cali (`mc_barrios`)

Análisis exploratorio del shapefile oficial de **manzanas censales / barrios** de Santiago de Cali.

| Fuente | `data/mc_barrios/mc_barrios.shp` (+ `.dbf`, `.prj`, `.shx`) |
|--------|-------------------------------------------------------------|
| CRS original | EPSG:6249 (MAGNA-SIRGAS / Cali-Valle 2009) |
| Registros | 339 polígonos (barrios) en 22 comunas |

**Columnas:** `id_barrio`, `barrio`, `comuna`, `shape_leng`, `shape_area`, `geometry`

In [ ]:
# @title Dependencias
%pip install -q geopandas pyogrio matplotlib seaborn pandas

In [ ]:
# @title Rutas y carga del shapefile
from pathlib import Path
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=0.95)

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data" / "mc_barrios").is_dir():
    REPO_ROOT = Path.cwd().parent
assert (REPO_ROOT / "data" / "mc_barrios" / "mc_barrios.shp").is_file(), (
    "No se encontró data/mc_barrios/mc_barrios.shp — ejecuta desde la raíz del repo"
)

SHP_PATH = REPO_ROOT / "data" / "mc_barrios" / "mc_barrios.shp"
OUT_DIR = REPO_ROOT / "data" / "mc_barrios" / "eda"
OUT_DIR.mkdir(parents=True, exist_ok=True)

gdf = gpd.read_file(SHP_PATH)
gdf_wgs = gdf.to_crs(4326)

print(f"Registros: {len(gdf)} barrios")
print(f"CRS: {gdf.crs}")
print(f"Comunas: {gdf['comuna'].nunique()}")
print(f"Salida EDA: {OUT_DIR}")

## 1. Vista general del dataset

In [ ]:
# @title Esquema, tipos y primeras filas
print("=== Columnas ===")
print(gdf.dtypes)
print("\n=== Muestra (atributos) ===")
display(gdf.drop(columns="geometry").head(10))

print("\n=== Valores nulos ===")
print(gdf.drop(columns="geometry").isnull().sum())

print("\n=== Geometría ===")
print(f"Válidas: {gdf.geometry.is_valid.all()}")
print(f"Vacías: {int(gdf.geometry.is_empty.sum())}")
print(f"Tipos: {gdf.geom_type.value_counts().to_dict()}")

bounds = gdf_wgs.total_bounds
print(f"\nExtensión WGS84: lon [{bounds[0]:.4f}, {bounds[2]:.4f}] | lat [{bounds[1]:.4f}, {bounds[3]:.4f}]")

In [ ]:
# @title Estadísticas descriptivas (área y perímetro)
gdf["area_km2"] = gdf["shape_area"] / 1e6
gdf["perim_km"] = gdf["shape_leng"] / 1e3

stats = gdf[["shape_area", "shape_leng", "area_km2", "perim_km"]].describe().round(4)
display(stats)
stats.to_csv(OUT_DIR / "estadisticas_area_perimetro.csv")
print("Guardado:", OUT_DIR / "estadisticas_area_perimetro.csv")

## 2. Comunas — cuántos barrios y área por comuna

In [ ]:
# @title Tabla por comuna
por_comuna = (
    gdf.groupby("comuna", as_index=False)
    .agg(
        n_barrios=("barrio", "count"),
        area_total_km2=("area_km2", "sum"),
        area_media_km2=("area_km2", "mean"),
        area_max_km2=("area_km2", "max"),
    )
    .sort_values("comuna", key=lambda s: s.astype(int))
)
por_comuna["pct_barrios"] = (100.0 * por_comuna["n_barrios"] / len(gdf)).round(1)
por_comuna["pct_area"] = (100.0 * por_comuna["area_total_km2"] / gdf["area_km2"].sum()).round(1)

display(por_comuna)
por_comuna.to_csv(OUT_DIR / "resumen_por_comuna.csv", index=False)
print("Guardado:", OUT_DIR / "resumen_por_comuna.csv")

In [ ]:
# @title Gráficos — barrios y área por comuna
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

orden = por_comuna.sort_values("n_barrios", ascending=True)
axes[0].barh(orden["comuna"].astype(str), orden["n_barrios"], color="steelblue")
axes[0].set_xlabel("Número de barrios")
axes[0].set_ylabel("Comuna")
axes[0].set_title("Barrios por comuna")

orden_a = por_comuna.sort_values("area_total_km2", ascending=True)
axes[1].barh(orden_a["comuna"].astype(str), orden_a["area_total_km2"], color="seagreen")
axes[1].set_xlabel("Área total (km²)")
axes[1].set_title("Área agregada por comuna")

plt.tight_layout()
plt.savefig(OUT_DIR / "01_barrios_area_por_comuna.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Barrios — distribución de tamaños y outliers

In [ ]:
# @title Distribución de área por barrio
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(gdf["area_km2"], bins=40, ax=axes[0], color="steelblue")
axes[0].set_xlabel("Área (km²)")
axes[0].set_title("Histograma — área por barrio")

sns.boxplot(data=gdf, x="comuna", y="area_km2", ax=axes[1], order=sorted(gdf["comuna"].unique(), key=int))
axes[1].tick_params(axis="x", rotation=90, labelsize=7)
axes[1].set_title("Área por barrio dentro de cada comuna")

plt.tight_layout()
plt.savefig(OUT_DIR / "02_distribucion_area_barrios.png", dpi=150, bbox_inches="tight")
plt.show()

top_grandes = gdf.nlargest(10, "area_km2")[["id_barrio", "barrio", "comuna", "area_km2"]]
top_chicos = gdf.nsmallest(10, "area_km2")[["id_barrio", "barrio", "comuna", "area_km2"]]
print("=== 10 barrios más grandes (km²) ===")
display(top_grandes)
print("=== 10 barrios más pequeños (km²) ===")
display(top_chicos)

## 4. Mapas

In [ ]:
# @title Mapa — todos los barrios (WGS84)
fig, ax = plt.subplots(figsize=(10, 10))
gdf_wgs.plot(ax=ax, column="comuna", cmap="tab20", linewidth=0.3, edgecolor="white", legend=False)
ax.set_title("Barrios de Cali coloreados por comuna", fontsize=12)
ax.set_xlabel("Longitud")
ax.set_ylabel("Latitud")
ax.set_aspect("equal")
plt.tight_layout()
plt.savefig(OUT_DIR / "03_mapa_barrios_por_comuna.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# @title Mapa — comunas agregadas (dissolve)
comunas_gdf = gdf.dissolve(by="comuna", as_index=False)
comunas_wgs = comunas_gdf.to_crs(4326)

fig, ax = plt.subplots(figsize=(10, 10))
comunas_wgs.plot(
    ax=ax, column="comuna", cmap="Set3", linewidth=0.8,
    edgecolor="black", legend=True, legend_kwds={"loc": "upper left", "fontsize": 7},
)
for _, row in comunas_wgs.iterrows():
    c = row.geometry.centroid
    ax.annotate(str(row["comuna"]), (c.x, c.y), fontsize=7, ha="center")
ax.set_title("22 comunas de Cali (unión de barrios)", fontsize=12)
ax.set_aspect("equal")
plt.tight_layout()
plt.savefig(OUT_DIR / "04_mapa_comunas.png", dpi=150, bbox_inches="tight")
plt.show()

comunas_gdf[["comuna", "area_km2"]].groupby("comuna")["area_km2"].sum().to_csv(
    OUT_DIR / "area_comunas_dissolve.csv"
)

In [ ]:
# @title Mapa — área del barrio (choropleth)
fig, ax = plt.subplots(figsize=(10, 10))
gdf_wgs.plot(
    ax=ax, column="area_km2", cmap="YlOrRd", linewidth=0.2,
    edgecolor="gray", legend=True,
    legend_kwds={"label": "Área (km²)", "shrink": 0.6},
)
ax.set_title("Área por barrio (km²)", fontsize=12)
ax.set_aspect("equal")
plt.tight_layout()
plt.savefig(OUT_DIR / "05_mapa_area_barrios.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Calidad de datos y exportación

In [ ]:
# @title Chequeos de calidad + catálogo barrios
dup_id = gdf["id_barrio"].duplicated().sum()
dup_nombre = gdf["barrio"].duplicated().sum()
print(f"id_barrio duplicados: {dup_id} (esperado 0)")
print(f"Nombres de barrio duplicados: {dup_nombre} (esperado 0)")

catalogo = gdf.drop(columns="geometry")[[
    "id_barrio", "barrio", "comuna", "area_km2", "perim_km", "shape_area", "shape_leng"
]].sort_values(["comuna", "barrio"], key=lambda c: c.astype(str) if c.name == "comuna" else c)
catalogo.to_csv(OUT_DIR / "catalogo_barrios.csv", index=False)
print(f"Catálogo: {len(catalogo)} filas -> {OUT_DIR / 'catalogo_barrios.csv'}")

resumen = {
    "n_barrios": len(gdf),
    "n_comunas": int(gdf["comuna"].nunique()),
    "crs": str(gdf.crs),
    "area_total_km2": round(float(gdf["area_km2"].sum()), 3),
    "bounds_wgs84": {
        "min_lon": round(float(bounds[0]), 5),
        "min_lat": round(float(bounds[1]), 5),
        "max_lon": round(float(bounds[2]), 5),
        "max_lat": round(float(bounds[3]), 5),
    },
    "comuna_mas_barrios": por_comuna.loc[por_comuna["n_barrios"].idxmax(), "comuna"],
    "comuna_menos_barrios": por_comuna.loc[por_comuna["n_barrios"].idxmin(), "comuna"],
}
(OUT_DIR / "resumen_eda.json").write_text(json.dumps(resumen, indent=2, ensure_ascii=False), encoding="utf-8")
print("\n=== Resumen EDA ===")
for k, v in resumen.items():
    print(f"  {k}: {v}")
print(f"\nArchivos en: {OUT_DIR}")